# 🎀 DecodeMatrimony on Colab ✨
Run the cells **top to bottom**. You need three files from your computer:
1. `decodematrimony.zip` (latest version from the chat)
2. `hies_roster.csv` (from `decodematrimony\data\raw\`)
3. `pakistan_cpi_2026.csv` (your CPI file)

SQL Server is not available here, so this notebook runs with `--no-db`. Colab files disappear when the session ends, so download the results at the end.

## 1. Upload the three files

In [ ]:
from google.colab import files
uploaded = files.upload()   # select all three files in the dialog
print(list(uploaded))

## 2. Unpack and place the files

In [ ]:
import os
for need in ["decodematrimony.zip", "hies_roster.csv", "pakistan_cpi_2026.csv"]:
    assert os.path.exists(f"/content/{need}"), f"Missing {need}: rerun the upload cell and select all three files"
!unzip -q -o /content/decodematrimony.zip -d /content
%cd /content/decodematrimony
!mkdir -p data/raw out
!cp /content/hies_roster.csv data/raw/hies_roster.csv
!cp /content/pakistan_cpi_2026.csv data/raw/pakistan_cpi_2026.csv
!ls data/raw data/reference config

## 3. Install what Colab lacks

In [ ]:
!pip -q install nltk sqlalchemy pytest

## 4. Run the unit tests (expect: 32 passed)

In [ ]:
!PYTHONPATH=src python -m pytest -q

## 5. Run the pipeline (about a minute or more with 1000 bootstrap draws)

In [ ]:
!PYTHONPATH=src DM_N_BOOT=1000 python -m decodematrimony.pipeline --roster data/raw/hies_roster.csv --roster-map config/hies_roster_map.json --year 2024 --no-db --gold-csv data/reference/gold_month_end_apgjsa_2026.csv --gold-cols Month Gold_Price_Per_Tola_24K --cpi-csv data/raw/pakistan_cpi_2026.csv --cpi-cols Month National_CPI_Index --base-month 2026-01

## 6. Check the Mahr benchmark (October row should show gold_index 80.05, cpi_index 108.26, gold_to_cpi_ratio about 73.94)

In [ ]:
import pandas as pd
b = pd.read_csv("out/mahr_benchmark.csv")
print(b.round(2).to_string(index=False))
print()
print("gold_index OK:", round(b["gold_index"].iloc[-1], 2) == 80.05)

## 7. Build the results page (`site/index.html`) from this run's results

In [ ]:
!python scripts/build_site.py
!ls -la site out

## 8. Look at the results table

In [ ]:
r = pd.read_csv("out/results.csv")
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 100)
print(r[["metric","group_label","estimate","ci_low","ci_high","p_value","p_adjusted","n_unweighted"]].round(4).to_string(index=False))

## 9. Download everything (results + the finished page)

In [ ]:
!zip -rq results.zip out site/index.html
files.download("results.zip")